### Setup envs

In [1]:
import os
import logging
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

### Model definition

In [2]:
class UserModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_genders'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_genders']) + 1, 4),
        ])

        self.lang_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_langs'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_langs']) + 1, 10),
        ])

        self.country_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_countries'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_countries']) + 1, 10),
        ])

        self.network_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_networks'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_networks']) + 1, 4),
        ])

        age_boundaries = np.array(conf['age_boundaries'])
        self.viewer_age_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.Discretization(age_boundaries.tolist()),
            tf.keras.layers.Embedding(len(age_boundaries), 2)
        ])

        self.centroids = tf.constant(conf['centroids'])
        self.viewer_lat_long_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.TextVectorization(
                standardize = None, split = self.classify,
                vocabulary = [str(i) for i in range(len(self.centroids))],
                max_tokens=len(self.centroids) + 2
                ),
            tf.keras.layers.Embedding(len(self.centroids) + 2, 2),
        ])

    @tf.function()
    def call(self, inputs):
        return tf.concat([
            self.gender_embedding(inputs["viewer_gender"]),
            self.lang_embedding(inputs["viewer_lang"]),
            self.country_embedding(inputs["viewer_country"]),
            self.network_embedding(inputs["viewer_network"]),
            self.viewer_age_embedding(inputs["viewer_age"]),
            self.viewer_lat_long_embedding(inputs["viewer_lat_long"]),
        ], axis = 1)

    @tf.keras.utils.register_keras_serializable()
    def classify(self, pair):
        """
        given a datapoint, compute the cluster closest to the datapoint. Return the cluster ID of that cluster.
        :param pair:
        :return: cluster ID
        """
        str_data = tf.strings.split(pair, sep = ",").values
        str_data = tf.map_fn(lambda x: tf.strings.regex_replace(x, "b'", ""), str_data)
        datapoints = tf.map_fn(lambda x: tf.strings.to_number(x), str_data, dtype = (tf.float32))
        datapoints = tf.reshape(datapoints, [-1, 2])

        expanded_centroids = tf.expand_dims(self.centroids, 1)
        expanded_vectors = tf.expand_dims(datapoints, 0)
        distances = tf.reduce_sum(tf.square(tf.subtract(expanded_vectors, expanded_centroids)), 2)
        clusters = tf.math.argmin(distances)
        return tf.strings.as_string(clusters)

In [3]:
class QueryModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		# We first use the user model for generating embeddings.
		self.embedding_model = UserModel(conf)
		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

In [4]:
class BroadcasterModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.broadcaster_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_broadcasters'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_broadcasters']) + 1, conf['broadcaster_embedding_dimension'])
        ])

    def call(self, broadcaster):
        return tf.concat([
            self.broadcaster_embedding(broadcaster),
        ], axis=1)

In [5]:
class CandidateModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		self.embedding_model = BroadcasterModel(conf)

		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dropout(0.5),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

### Load data

In [6]:
def load_data_file_cold(file, stats):
    print('loading file:' + file)
    training_df = pd.read_csv(
        file,
        skiprows=[0],
        names=["viewer",
               "broadcaster",
               "viewer_age",
               "viewer_gender",
               "viewer_longitude",
               "viewer_latitude",
               "viewer_lang",
               "viewer_country",
               "broadcaster_age",
               "broadcaster_gender",
               "broadcaster_longitude",
               "broadcaster_latitude",
               "broadcaster_lang",
               "broadcaster_country",
               "duration", 
               "viewer_network", 
               "broadcaster_network", 
               "count"], 
        dtype={
            'viewer': np.unicode,
            'broadcaster': np.unicode,
            'viewer_age': np.single,
            'viewer_gender': np.unicode,
            'viewer_longitude': np.single,
            'viewer_latitude': np.single,
            'viewer_lang': np.unicode,
            'viewer_country': np.unicode,
            'broadcaster_age': np.single,
            'broadcaster_longitude': np.single,
            'broadcaster_latitude': np.single,
            'broadcaster_lang': np.unicode,
            'broadcaster_country': np.unicode,
            'duration': np.single,
            'viewer_network': np.unicode,
            'broadcaster_network': np.unicode,
            'count': np.unicode,
        })

    values = {
        'viewer': 'unknown',
        'broadcaster': 'unknown',
        'viewer_age': 30,
        'viewer_gender': 'unknown',
        'viewer_longitude': 0,
        'viewer_latitude': 0,
        'viewer_lang': 'unknown',
        'viewer_country': 'unknown',
        'broadcaster_age': 30,
        'broadcaster_longitude': 0,
        'broadcaster_latitude': 0,
        'broadcaster_lang': 'unknown',
        'broadcaster_country': 'unknown',
        'duration': 0,
        'viewer_network': 'unknown',
        'broadcaster_network': 'unknown',
        "viewer_lat_long": tf.constant(["40.36393,-74.89611"]),
        'count': '1'
    }

    training_df = training_df.sample(frac=0.1)
    training_df.fillna(value=values, inplace=True)
    training_df['viewer_lat_long'] = training_df[['viewer_latitude', 'viewer_longitude']].apply(lambda x: '{},{}'.format(x[0],x[1]), axis=1)
    training_df['duration'] = np.log(1 + training_df['duration'])
    print(training_df.head(10))
    print(training_df.iloc[-10:])
    # stats.send_stats('data-size', len(training_df.index))
    return training_df


def load_training_data_cold(file, stats):
    ratings_df = load_data_file_cold(file, stats)
    print('creating data set')
    training_ds = (
        tf.data.Dataset.from_tensor_slices(
            ({
                "viewer": tf.cast(
                    ratings_df['viewer'].values,
                    tf.string),
                "viewer_gender": tf.cast(
                    ratings_df['viewer_gender'].values,
                    tf.string),
                "viewer_lang": tf.cast(
                    ratings_df['viewer_lang'].values,
                    tf.string),
                "viewer_country": tf.cast(
                    ratings_df['viewer_country'].values,
                    tf.string),
                "viewer_age": tf.cast(
                    ratings_df['viewer_age'].values,
                    tf.int32),
                "viewer_longitude": tf.cast(
                    ratings_df['viewer_longitude'].values,
                    tf.float16),
                "viewer_latitude": tf.cast(
                    ratings_df['viewer_latitude'].values,
                    tf.float16),
                "broadcaster": tf.cast(
                    ratings_df['broadcaster'].values,
                    tf.string),
                "viewer_network": tf.cast(
                    ratings_df['viewer_network'].values,
                    tf.string),
                "broadcaster_network": tf.cast(
                    ratings_df['broadcaster_network'].values,
                    tf.string),
                "duration": tf.cast(
                    ratings_df['duration'].values,
                    tf.float16),
                "viewer_lat_long": tf.cast(
                    ratings_df['viewer_lat_long'].values,
                    tf.string),
            })))

    return training_ds
            

def prepare_training_data_cold(train_ds):
    print('prepare_training_data')
    training_ds = train_ds.cache().map(lambda x: {
        "broadcaster": x["broadcaster"],
        "viewer": x["viewer"],
        "viewer_gender": x["viewer_gender"],
        "viewer_lang": x["viewer_lang"],
        "viewer_country": x["viewer_country"],
        "viewer_age": x["viewer_age"],
        "viewer_longitude": x["viewer_longitude"],
        "viewer_latitude": x["viewer_latitude"],
        "viewer_network": x["viewer_network"],
        "broadcaster_network": x["broadcaster_network"],
        "duration": x["duration"],
        "viewer_lat_long": x["viewer_lat_long"],
    }, num_parallel_calls=tf.data.AUTOTUNE,
       deterministic=False)

    print('done prepare_training_data')
    return training_ds

In [7]:
def get_broadcaster_data_set(train_ds):
    broadcasters = train_ds.cache().map(lambda x: x["broadcaster"], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    broadcasters_ds = tf.data.Dataset.from_tensor_slices(
        np.unique(list(broadcasters.as_numpy_iterator())))
    return broadcasters_ds

In [8]:
training_dataset = load_training_data_cold("csv/2022-01-12.csv", "")

loading file:csv/2022-01-12.csv
                                                   viewer  \
4916411   07 01 51 07 78 f4 6c e0 a9 c0 46 a4 d6 fa c5 ce   
3930561   39 97 35 6c 17 76 dc 28 9c 48 95 6d 16 bb 0f 2d   
4456431   5c 87 03 54 48 81 34 f3 4c 93 8c 00 ca cc c1 0c   
9696199   85 98 35 5a 85 df b4 61 22 c6 27 d9 0a c7 c3 2c   
2747932   8c d3 d3 2a 0c 95 c5 19 48 db 7c 2e 11 30 4d 3a   
6610422   9b 19 8e 72 bd 50 c5 de 6c 06 51 27 44 20 5f 10   
3538124   14 56 ef 05 1a ab 8f 56 9b 5f 1e e1 c4 95 cb 01   
854980    f3 c3 a3 d2 27 79 50 88 65 d6 53 7c c9 a8 07 a5   
10250820  1c e8 b2 83 c7 e1 91 a1 b0 f1 e0 b4 f9 ba 93 7d   
5011131   8f 7c f5 7e 7d ff 10 7e 10 fd 22 16 43 50 a8 04   

                                              broadcaster  viewer_age  \
4916411   e1 8d 6c 50 77 cc 5c 21 3e 59 76 ca 45 05 3a 4b        27.0   
3930561   3c 26 6f ee 90 51 ae d2 e7 6d bf 24 65 b4 d9 1e        61.0   
4456431   33 1a 68 0e 0c 75 e6 78 4d 1a 14 28 5b 04 15 41        29.0   
9696

In [9]:
train = prepare_training_data_cold(training_dataset)

prepare_training_data
done prepare_training_data


In [10]:
broadcasters_data_set = get_broadcaster_data_set(training_dataset)

### Prepare model conf

In [11]:
def get_list(training_data, key):
    return training_data.batch(1_000_000).map(lambda x: x[key], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)


def get_unique_list(data):
    return np.unique(np.concatenate(list(data)))

In [12]:
user_genders = get_list(train, 'viewer_gender')

In [13]:
user_langs = get_list(train, 'viewer_lang')

In [14]:
user_countries = get_list(train, 'viewer_country')

In [15]:
viewer_age = get_list(train, 'viewer_age')

In [16]:
user_networks = get_list(train, 'viewer_network')

### derive input dims

In [17]:
unique_user_genders = get_unique_list(user_genders)

In [18]:
len(unique_user_genders)

3

In [19]:
unique_user_langs = get_unique_list(user_langs)

In [20]:
len(unique_user_langs)

63

In [21]:
unique_user_countries = get_unique_list(user_countries)

In [22]:
len(unique_user_countries)

177

In [23]:
unique_user_networks = get_unique_list(user_networks)

In [24]:
len(unique_user_networks)

5

In [25]:
broadcaster_ids = get_list(train, 'broadcaster')

In [26]:
unique_broadcasters = get_unique_list(broadcaster_ids)

In [27]:
len(unique_broadcasters)

81303

In [28]:
broadcaster_embedding_dimension = 32

In [29]:
cold_start_conf = {
    'unique_genders': unique_user_genders,
    'unique_langs': unique_user_langs,
    'unique_countries': unique_user_countries,
    'unique_networks': unique_user_networks,
    'unique_broadcasters': unique_broadcasters,
    'broadcaster_embedding_dimension': broadcaster_embedding_dimension,
    'age_boundaries': [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")],
    'centroids': [[36.68147669256268, -82.8910274009993],
        [23.22243322909555, 78.23027450833709],
        [50.04997682638993, 0.22379313938744885],
        [37.9309447099281, -117.00741350764692],
        [-32.795864819917725, 148.7159172660312],
        [-18.570548393114084, -54.280255665692565],
        [13.921140442819565, 116.38740315555172],
        [29.78951080730802, 40.279515865947936]]
}

In [30]:
cold_start_conf

{'unique_genders': array([b'female', b'male', b'unknown'], dtype=object),
 'unique_langs': array([b'ar', b'az', b'bg', b'bm', b'bn', b'bs', b'ca', b'co', b'cs',
        b'da', b'de', b'el', b'en', b'es', b'eu', b'fa', b'fi', b'fr',
        b'ga', b'gl', b'gu', b'he', b'hi', b'hr', b'hu', b'id', b'in',
        b'it', b'iw', b'ja', b'ka', b'ko', b'lt', b'lv', b'ml', b'mr',
        b'ms', b'nb', b'ne', b'nl', b'ny', b'pa', b'pl', b'ps', b'pt',
        b'ro', b'ru', b'sk', b'sl', b'sm', b'sq', b'sr', b'sv', b'ta',
        b'te', b'th', b'ti', b'tr', b'uk', b'ur', b'uz', b'vi', b'zh'],
       dtype=object),
 'unique_countries': array([b'419', b'AD', b'AE', b'AF', b'AG', b'AL', b'AO', b'AQ', b'AR',
        b'AS', b'AT', b'AU', b'AW', b'AX', b'AZ', b'BA', b'BD', b'BE',
        b'BF', b'BG', b'BH', b'BJ', b'BN', b'BO', b'BQ', b'BR', b'BS',
        b'BT', b'BY', b'BZ', b'CA', b'CF', b'CH', b'CI', b'CL', b'CN',
        b'CO', b'CR', b'CV', b'CY', b'CZ', b'DE', b'DK', b'DO', b'DZ',
        b'EC',

### query model

In [31]:
query_model = QueryModel(cold_start_conf)

### broadcaster model

In [32]:
candidate_model = CandidateModel(cold_start_conf)

### Candidate / Ranking model

In [33]:
class RankingModel(tf.keras.Model):

	def __init__(self):
		super().__init__()
		embedding_dimension = 32

		# Compute predictions.
		self.ratings = tf.keras.Sequential(
			[
				# Learn multiple dense layers.
				tf.keras.layers.Dense(256, activation = "relu"),
				tf.keras.layers.Dense(64, activation = "relu"),
				# Make rating predictions in the final layer.
				tf.keras.layers.Dense(1)
			]
		)

	def call(self, inputs):
		query_embeddings, positive_broadcaster_embeddings = inputs
		return self.ratings(tf.concat([query_embeddings, positive_broadcaster_embeddings], axis = 1))

In [34]:
ranking_model = RankingModel()

### Loss and metrics

In [35]:
ranking_task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

In [36]:
retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=broadcasters_data_set.batch(128).map(candidate_model)
    )
)

In [37]:
from typing import Dict, Text

In [38]:
class TwoTowers(tfrs.models.Model) :

    def __init__(self, candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight):
        super().__init__()
        
        self.query_model: tf.keras.Model = query_model
        self.candidate_model: tf.keras.Model = candidate_model
        self.ranking_model: tf.keras.Model = ranking_model
        self.ranking_task = ranking_task
        self.retrieval_task = retrieval_task
        
        # The loss weights.
        self.ranking_weight = ranking_weight
        self.retrieval_weight = retrieval_weight
    
    def train_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        with tf.GradientTape() as tape:
            query_embeddings = self.query_model({
                "viewer_gender": features["viewer_gender"],
                "viewer_lang": features["viewer_lang"],
                "viewer_country": features["viewer_country"],
                "viewer_age": features["viewer_age"],
                "viewer_network": features["viewer_network"],
                "viewer_latitude": features["viewer_latitude"],
                "viewer_longitude": features["viewer_longitude"],
                "viewer_lat_long": features["viewer_lat_long"],
            })
            positive_broadcaster_embeddings = self.candidate_model(
                features["broadcaster"])
            
            labels = features["duration"]
            ranking_predictions = self.ranking_model(
                (query_embeddings, positive_broadcaster_embeddings)
            )
            ranking_loss = self.ranking_task(
                labels = labels,
                predictions = ranking_predictions,
            )
            
            retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)
            
            regularization_loss = sum(self.losses)
            
            total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss 
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        
        return metrics

    def test_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        labels = features["duration"]
        
        query_embeddings = self.query_model({
            "viewer_gender": features["viewer_gender"],
            "viewer_lang": features["viewer_lang"],
            "viewer_country": features["viewer_country"],
            "viewer_age": features["viewer_age"],
            "viewer_network": features["viewer_network"],
            "viewer_latitude": features["viewer_latitude"],
            "viewer_longitude": features["viewer_longitude"],
            "viewer_lat_long": features["viewer_lat_long"],
        })
        positive_broadcaster_embeddings = self.candidate_model(
            features["broadcaster"])
        
        rating_predictions = self.ranking_model(
            (query_embeddings, positive_broadcaster_embeddings)
        )
        
        retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)

        # The task computes the loss and the metrics.
        ranking_loss = self.ranking_task(labels = labels, predictions = rating_predictions)
        
        regularization_loss = sum(self.losses)
        
        total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss + retrieval_loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        return metrics        

### Retrieval-specialized model

In [39]:
ranking_weight = 0
retrieval_weight = 1

In [40]:
model = TwoTowers(candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight)

In [41]:
learning_rate = 0.05
batch_size = 16384
epochs = 20
patience = 2
top_k = 1999

In [42]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate),
    run_eagerly=True)

In [43]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=True)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(batch_size).cache()

In [44]:
# model.fit(train_ds, epochs=epochs)
callback = tf.keras.callbacks.EarlyStopping(monitor = "total_loss", 
    patience = patience,
    verbose=1,
    restore_best_weights=True)
hist = model.fit(
    cached_train,
    validation_data=cached_test,
    validation_freq=1,
    epochs=epochs,
    callbacks = [callback])

Epoch 1/20
Instructions for updating:
Use fn_output_signature instead
Instructions for updating:
The `validate_indices` argument has no effect. Indices are always validated on CPU and never validated on GPU.
5/5 [==============================] - 406s 88s/step - root_mean_squared_error: 4.5148 - factorized_top_k/top_1_categorical_accuracy: 2.2500e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0018 - factorized_top_k/top_10_categorical_accuracy: 0.0032 - factorized_top_k/top_50_categorical_accuracy: 0.0098 - factorized_top_k/top_100_categorical_accuracy: 0.0143 - loss: 20.1085 - regularization_loss: 0.0069 - total_loss: 152538.8490 - val_root_mean_squared_error: 4.5324 - val_factorized_top_k/top_1_categorical_accuracy: 2.5000e-04 - val_factorized_top_k/top_5_categorical_accuracy: 0.0013 - val_factorized_top_k/top_10_categorical_accuracy: 0.0019 - val_factorized_top_k/top_50_categorical_accuracy: 0.0054 - val_factorized_top_k/top_100_categorical_accuracy: 0.0077 - val_loss: 29706.1

5/5 [==============================] - 799s 56s/step - root_mean_squared_error: 4.4599 - factorized_top_k/top_1_categorical_accuracy: 4.6250e-04 - factorized_top_k/top_5_categorical_accuracy: 0.0030 - factorized_top_k/top_10_categorical_accuracy: 0.0061 - factorized_top_k/top_50_categorical_accuracy: 0.0253 - factorized_top_k/top_100_categorical_accuracy: 0.0461 - loss: 19.8801 - regularization_loss: 0.0128 - total_loss: 132730.1562 - val_root_mean_squared_error: 4.4581 - val_factorized_top_k/top_1_categorical_accuracy: 3.5000e-04 - val_factorized_top_k/top_5_categorical_accuracy: 0.0026 - val_factorized_top_k/top_10_categorical_accuracy: 0.0049 - val_factorized_top_k/top_50_categorical_accuracy: 0.0228 - val_factorized_top_k/top_100_categorical_accuracy: 0.0408 - val_loss: 25954.8047 - val_regularization_loss: 0.0130 - val_total_loss: 25934.8223
Epoch 18/20
5/5 [==============================] - 342s 71s/step - root_mean_squared_error: 4.4503 - factorized_top_k/top_1_categorical_accur

In [45]:
hist.history

{'root_mean_squared_error': [4.5148491859436035,
  4.541823387145996,
  4.527632236480713,
  4.5591816902160645,
  4.539393424987793,
  4.52884578704834,
  4.502435684204102,
  4.4967546463012695,
  4.4870991706848145,
  4.486034393310547,
  4.472365856170654,
  4.461777210235596,
  4.482734203338623,
  4.479116439819336,
  4.457374095916748,
  4.457401752471924,
  4.4599385261535645,
  4.4502716064453125,
  4.431159496307373,
  4.443517208099365],
 'factorized_top_k/top_1_categorical_accuracy': [0.00022499999613501132,
  0.00043750001350417733,
  0.000375000003259629,
  0.0002749999985098839,
  0.00016250000044237822,
  0.0003124999930150807,
  0.00022499999613501132,
  0.0002749999985098839,
  0.00038750001112930477,
  0.00032500000088475645,
  0.00028750000637955964,
  0.00038750001112930477,
  0.00044999999227002263,
  0.00038750001112930477,
  0.0004124999977648258,
  0.00048749998677521944,
  0.0004625000001396984,
  0.0005624999757856131,
  0.0005750000127591193,
  0.00076249998

### Ranking Specialized Model

In [46]:
ranking_weight = 1
retrieval_weight = 0

In [47]:
model = TwoTowers(candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight)

In [48]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=True)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(batch_size).cache()

In [49]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate),
    run_eagerly=True)

In [50]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='total_loss', 
    patience=patience,
    verbose=1,
    restore_best_weights=True
)
new_model_history = model.fit(
    cached_train,
    validation_data=cached_test,
    validation_freq=5,
    epochs=epochs,
    verbose=0)

In [51]:
new_model_history.history

{'root_mean_squared_error': [5.936650276184082,
  2.4628472328186035,
  1.5625131130218506,
  1.5573855638504028,
  1.5808981657028198,
  1.5205504894256592,
  1.4746445417404175,
  1.4704691171646118,
  1.4611566066741943,
  1.4675021171569824,
  1.4545657634735107,
  1.4491511583328247,
  1.447880506515503,
  1.446402907371521,
  1.4439936876296997,
  1.4463474750518799,
  1.4453115463256836,
  1.4395310878753662,
  1.4360432624816895,
  1.4417520761489868],
 'factorized_top_k/top_1_categorical_accuracy': [0.00038750001112930477,
  9.999999747378752e-05,
  0.0001500000071246177,
  0.00013749999925494194,
  0.0002125000028172508,
  0.00019999999494757503,
  0.0003124999930150807,
  0.0001500000071246177,
  0.00016250000044237822,
  0.00016250000044237822,
  0.00022499999613501132,
  0.00016250000044237822,
  0.00022499999613501132,
  0.00023750000400468707,
  0.0002125000028172508,
  0.00019999999494757503,
  0.00019999999494757503,
  0.00022499999613501132,
  0.00022499999613501132,


### Joint Model

In [52]:
ranking_weight = 1
retrieval_weight = 1

In [53]:
model = TwoTowers(candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight)

In [54]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=True)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(batch_size).cache()

In [55]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate),
    run_eagerly=True)

In [56]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor='total_loss', 
    patience=patience,
    verbose=1,
    restore_best_weights=True
)
new_model_history = model.fit(
    cached_train,
    validation_data=cached_test,
    validation_freq=5,
    epochs=epochs,
    verbose=0)

In [57]:
new_model_history.history

{'root_mean_squared_error': [2.4469094276428223,
  1.4682605266571045,
  1.4553033113479614,
  1.4462469816207886,
  1.4517507553100586,
  1.48703932762146,
  1.4607218503952026,
  1.4595630168914795,
  1.5350983142852783,
  1.4938925504684448,
  1.450595736503601,
  1.4493255615234375,
  1.4658397436141968,
  1.4607800245285034,
  1.450001835823059,
  1.4562311172485352,
  1.4751707315444946,
  1.4645057916641235,
  1.4485427141189575,
  1.4688162803649902],
 'factorized_top_k/top_1_categorical_accuracy': [0.0001500000071246177,
  0.00039999998989515007,
  0.0007249999907799065,
  0.0007624999852851033,
  0.0009374999790452421,
  0.0010000000474974513,
  0.0011500000255182385,
  0.0011375000467523932,
  0.0010499999625608325,
  0.001537499949336052,
  0.0016499999910593033,
  0.0014875000342726707,
  0.001587499980814755,
  0.0017750000115484,
  0.002237499924376607,
  0.0020749999675899744,
  0.0021375000942498446,
  0.0022750000935047865,
  0.0026000000070780516,
  0.002637499943375

In [58]:
accuracy = new_model_history.history["factorized_top_k/top_100_categorical_accuracy"][-1]
print(f"Retrieval top-100 accuracy: {accuracy:.4f}.")

Retrieval top-100 accuracy: 0.0968.


In [59]:
root_mean_squared_error = new_model_history.history["root_mean_squared_error"][-1]
print(f"Ranking RMSE: {root_mean_squared_error:.4f}.")

Ranking RMSE: 1.4688.
